# Tutorial: Exploratory Data Analysis & Visualization

> **Type:** Tutorial. Follow along step by step.  
> **Time:** ~2 hours  
> **Prerequisites:** Modules 06–07 (pandas, data cleaning)  
> **What you'll build:** A complete, structured EDA of the Titanic dataset, from raw inspection to a 5-bullet story that informs feature engineering.

---

## What We're Building

We'll run a structured EDA on the Titanic passenger dataset. By the end, you'll understand every variable visually and statistically, know which features look predictive, and have a written summary of findings ready to drive the next step (feature engineering).

The same structure (inspect → univariate → bivariate → multivariate → summarise) works on any dataset.

## Step 0: Setup

In [ ]:
# !pip install pandas matplotlib seaborn scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', 20)
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')
np.random.seed(42)

print('Ready.')

## Step 1: Load and Inspect — The 5-Command Checklist

In [ ]:
df = sns.load_dataset('titanic')
print(f'Shape: {df.shape}  →  {df.shape[0]} passengers, {df.shape[1]} columns')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Missing values: visualised as a heatmap
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_summary = pd.DataFrame({'count': missing, 'pct': missing_pct}).sort_values('count', ascending=False)
print('Missing values:')
print(missing_summary[missing_summary['count'] > 0])

In [ ]:
# Visual missing-value map
fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(df.isnull(), yticklabels=False, cbar=False,
            cmap='viridis', ax=ax)
ax.set_title('Missing Value Map (yellow = missing)', fontweight='bold')
ax.set_xlabel('Column')
plt.tight_layout()
plt.savefig('eda_missing.png', dpi=120, bbox_inches='tight')
plt.show()

**Chart description.** A missing-value map of the Titanic data. Each column is a feature and each thin row is a passenger. Yellow marks a missing value. Two columns stand out: deck is mostly yellow (77% missing) and age has scattered gaps (about 20% missing). The embarked and embark_town columns are missing for 2 passengers, too few to see. Every other column is complete.

> **What just happened?** Three things stand out: `age` is missing for 19% of passengers, `deck` for 77%, and `embark_town`/`embarked` for 2 rows. These findings drive the cleaning and feature engineering decisions.

---

## Step 2: Target Variable Analysis — Who Survived?

Always start with the thing you're trying to predict.

In [ ]:
survival_counts = df['survived'].value_counts()
survival_rate   = df['survived'].mean()

print(f'Survived:     {survival_counts[1]} ({survival_rate:.1%})')
print(f'Did not:      {survival_counts[0]} ({1-survival_rate:.1%})')
print(f'\nClass imbalance: {survival_counts[0]/survival_counts[1]:.1f}:1 (negative:positive)')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(['Did not survive', 'Survived'], survival_counts.values,
            color=['#E15759', '#59A14F'], edgecolor='white')
axes[0].set_title('Survival Counts', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(survival_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

axes[1].pie(survival_counts.values, labels=['Did not survive', 'Survived'],
            colors=['#E15759', '#59A14F'], autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Survival Rate', fontweight='bold')

plt.tight_layout()
plt.savefig('eda_target.png', dpi=120, bbox_inches='tight')
plt.show()

**Chart description.** The target. Left: a bar chart of counts, 549 passengers who didn't survive and 342 who did. Right: the same as a pie chart, 61.6% and 38.4%.

---

## Step 3: Univariate Analysis — Each Feature in Isolation

### 3a. Numeric Features

In [ ]:
numeric_cols = ['age', 'fare', 'sibsp', 'parch']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, col in enumerate(numeric_cols):
    data = df[col].dropna()
    
    # Histogram
    axes[0, i].hist(data, bins=25, color='#4E79A7', edgecolor='white')
    axes[0, i].axvline(data.mean(),   color='red',    linestyle='--', label=f'Mean={data.mean():.1f}')
    axes[0, i].axvline(data.median(), color='orange', linestyle='--', label=f'Median={data.median():.1f}')
    axes[0, i].set_title(col, fontweight='bold')
    axes[0, i].legend(fontsize=7)
    
    # Box plot
    axes[1, i].boxplot(data, vert=True, patch_artist=True,
                       boxprops={'facecolor': '#4E79A7', 'alpha': 0.7})
    axes[1, i].set_title(f'{col} (boxplot)', fontweight='bold')
    
    skew = data.skew()
    axes[0, i].set_xlabel(f'skewness: {skew:.2f}')

plt.suptitle('Univariate Analysis — Numeric Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_univariate_numeric.png', dpi=120, bbox_inches='tight')
plt.show()

**Chart description.** Histograms (top) and boxplots (bottom) of four numeric features, with dashed lines for the mean and median. Age is roughly bell-shaped around 30, with a skewness of 0.39. Fare is heavily right-skewed (skewness 4.79): most fares are under 50, and a few reach about 500. Siblings or spouses aboard (sibsp) and parents or children aboard (parch) are mostly 0, with a few large values that the boxplots show as outliers.

In [ ]:
# Quantify the skew the histograms above show visually
print(f"fare skewness: {df['fare'].skew():.2f}")
print(f"age skewness:  {df['age'].skew():.2f}")

> **Observations:** `fare` is heavily right-skewed (a few very expensive tickets). `age` is roughly normal, with a slight right skew. `sibsp` and `parch` are mostly zero (most passengers travelled alone). These suggest log-transforming `fare` as a feature engineering step.

### 3b. Categorical Features

In [ ]:
cat_cols = ['pclass', 'sex', 'embarked', 'class', 'who', 'alone']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    counts = df[col].value_counts()
    axes[i].bar(counts.index.astype(str), counts.values,
                color=sns.color_palette('Set2', len(counts)),
                edgecolor='white')
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=15)
    for j, v in enumerate(counts.values):
        axes[i].text(j, v + 2, str(v), ha='center', fontsize=9)

plt.suptitle('Univariate Analysis — Categorical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_univariate_categorical.png', dpi=120, bbox_inches='tight')
plt.show()

**Chart description.** Six bar charts of category counts. Class: 491 third, 216 first and 184 second. Sex: 577 male and 314 female. Embarked: 644 from Southampton, 168 from Cherbourg and 77 from Queenstown. Who: 537 men, 271 women and 83 children. Alone: 537 travelling alone and 354 not.

---

## Step 4: Bivariate Analysis — Features vs. Target

In [ ]:
# Survival rate by key categorical features
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (col, ax) in enumerate(zip(['sex', 'pclass', 'embarked'], axes)):
    survival_by = df.groupby(col)['survived'].mean().sort_values(ascending=False)
    bars = ax.bar(survival_by.index.astype(str), survival_by.values,
                  color=sns.color_palette('Set2', len(survival_by)),
                  edgecolor='white')
    ax.set_title(f'Survival Rate by {col}', fontweight='bold')
    ax.set_ylabel('Survival Rate')
    ax.set_ylim(0, 1)
    ax.axhline(df['survived'].mean(), color='black', linestyle='--',
               linewidth=1, label=f'Overall: {df["survived"].mean():.0%}')
    ax.legend(fontsize=9)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{bar.get_height():.0%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('eda_bivariate_categorical.png', dpi=120, bbox_inches='tight')
plt.show()

**Chart description.** Survival rate by sex, class and port, with a dashed line for the overall rate of 38%. Women survive at 74% and men at 19%. First class at 63%, second at 47% and third at 24%. Cherbourg at 55%, Queenstown at 39% and Southampton at 34%.

In [ ]:
# Age and Fare distributions by survival
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age
for survived, label, colour in [(0, 'Did not survive', '#E15759'), (1, 'Survived', '#59A14F')]:
    sns.kdeplot(data=df[df['survived'] == survived]['age'].dropna(),
                ax=axes[0], label=label, color=colour, fill=True, alpha=0.3, linewidth=2)
axes[0].set_title('Age Distribution by Survival', fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].legend()

# Fare (log scale for readability)
for survived, label, colour in [(0, 'Did not survive', '#E15759'), (1, 'Survived', '#59A14F')]:
    sns.kdeplot(data=np.log1p(df[df['survived'] == survived]['fare'].dropna()),
                ax=axes[1], label=label, color=colour, fill=True, alpha=0.3, linewidth=2)
axes[1].set_title('log(Fare+1) Distribution by Survival', fontweight='bold')
axes[1].set_xlabel('log(Fare + 1)')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_bivariate_numeric.png', dpi=120, bbox_inches='tight')
plt.show()

**Chart description.** Two density plots comparing survivors and non-survivors. Left, age: the curves mostly overlap, peaking in the twenties. Survivors have a bump below about 10, so young children survived more often. Right, the log of fare plus one: non-survivors peak sharply at low fares. Survivors spread further into higher fares.

In [ ]:
# Statistical test: are fare distributions different between survivors and non-survivors?
fare_survived = df[df['survived'] == 1]['fare'].dropna()
fare_died     = df[df['survived'] == 0]['fare'].dropna()

# Welch's test: the two groups have very different fare variances
t_stat, p_value = stats.ttest_ind(fare_survived, fare_died, equal_var=False)
print(f'Mean fare (survived):     £{fare_survived.mean():.2f}')
print(f'Mean fare (did not):      £{fare_died.mean():.2f}')
print(f'p-value: {p_value:.2e}  →  {"Significant" if p_value < 0.05 else "Not significant"}')

---

## Step 5: Multivariate Analysis

### 5a. Correlation Heatmap

In [ ]:
numeric_features = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']
corr = df[numeric_features].corr()

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))  # hide upper triangle (mirror)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

**Chart description.** A correlation heatmap of the numeric features, lower triangle only. Survival correlates with class at −0.34 and fare at 0.26. Class and fare correlate at −0.55, the strongest pair: first class costs more. Class and age correlate at −0.37. Siblings and parents aboard correlate at 0.41.

### 5b. Survival Rate: Class × Sex (Two-way comparison)

In [ ]:
pivot = df.pivot_table(values='survived', index='pclass', columns='sex', aggfunc='mean')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Heatmap
sns.heatmap(pivot, annot=True, fmt='.0%', cmap='RdYlGn', center=0.38,
            ax=axes[0], linewidths=0.5, vmin=0, vmax=1)
axes[0].set_title('Survival Rate by Class and Sex', fontweight='bold')

# Grouped bar
pivot.plot(kind='bar', ax=axes[1],
           color={'female': '#F28E2B', 'male': '#4E79A7'},
           edgecolor='white', rot=0)
axes[1].set_title('Survival Rate by Class and Sex', fontweight='bold')
axes[1].set_ylabel('Survival Rate')
axes[1].set_xlabel('Passenger Class')
axes[1].legend(title='Sex')

plt.tight_layout()
plt.savefig('eda_multivariate_class_sex.png', dpi=120, bbox_inches='tight')
plt.show()

**Chart description.** Survival rate by class and sex, as a heatmap (left) and a grouped bar chart (right). Women: 97% in first class, 92% in second and 50% in third. Men: 37%, 16% and 14%. Sex matters more than class, and the two combine.

### 5c. Pairplot — Full Feature View

In [ ]:
pair_data = df[['survived', 'pclass', 'age', 'fare', 'sibsp']].dropna()
pair_data['survived_label'] = pair_data['survived'].map({0: 'Did not survive', 1: 'Survived'})

g = sns.pairplot(
    pair_data,
    hue='survived_label',
    palette={'Did not survive': '#E15759', 'Survived': '#59A14F'},
    vars=['pclass', 'age', 'fare', 'sibsp'],
    plot_kws={'alpha': 0.4, 's': 20},
    diag_kind='kde'
)
g.fig.suptitle('Pairplot — Titanic Features by Survival', y=1.02, fontweight='bold')
plt.savefig('eda_pairplot.png', dpi=100, bbox_inches='tight')
plt.show()

**Chart description.** A pairplot of class, age, fare and sibsp, coloured by survival, with density curves on the diagonal. Third class is dominated by non-survivors and first class has more survivors. High fares are mostly survivors. Age and sibsp separate the groups only weakly.

---

## Step 6: Spotting Surprising Patterns

In [ ]:
# Age by class and survival: is the 'women and children first' effect visible?
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, pclass in enumerate([1, 2, 3]):
    class_data = df[df['pclass'] == pclass]
    for survived, label, colour in [(0, 'Died', '#E15759'), (1, 'Survived', '#59A14F')]:
        ages = class_data[class_data['survived'] == survived]['age'].dropna()
        axes[i].hist(ages, bins=20, alpha=0.5, label=label, color=colour,
                     edgecolor='white', density=True)
    axes[i].set_title(f'Class {pclass}: Age by Survival', fontweight='bold')
    axes[i].set_xlabel('Age')
    axes[i].legend()

plt.suptitle('Age Distribution by Class and Survival — "Children First" Effect',
             fontweight='bold')
plt.tight_layout()
plt.savefig('eda_surprising.png', dpi=120, bbox_inches='tight')
plt.show()

**Chart description.** Age histograms of passengers who died and survived, one panel per class. In every class, children under about 10 are mostly survivors. In first class, survivors spread across all ages. In third class, deaths outnumber survivals across most adult ages.

---

## Step 7: Write the EDA Summary

The most important output of EDA is a short list of findings that will drive modelling decisions.

In [ ]:
print("""
═══════════════════════════════════════════════════════
  TITANIC EDA SUMMARY — KEY FINDINGS
═══════════════════════════════════════════════════════

1. TARGET IMBALANCE
   Only 38% of passengers survived. Any model must be
   evaluated with metrics that account for imbalance
   (F1, AUC — not just accuracy).

2. SEX IS THE STRONGEST PREDICTOR
   Female survival rate: 74%. Male: 19%.
   The "women and children first" policy is clearly
   visible in the data.

3. CLASS MATTERS — AND INTERACTS WITH SEX
   1st-class passengers survived at 63% vs 24% in 3rd.
   The interaction (pclass × sex) is more predictive
   than either feature alone.

4. FARE IS HEAVILY SKEWED — LOG TRANSFORM IT
   Fare ranges from 0 to 512. Survivors paid more.
   Log-transform (log1p) will reduce skew and improve
   linear models.

5. AGE HAS 19% MISSING — IMPUTE WITH CARE
   Age is moderately predictive (children survived more
   in 1st/2nd class). Use median imputation by class,
   and add an 'age_was_missing' flag feature.

FEATURE ENGINEERING IDEAS (from this EDA):
  - family_size = sibsp + parch + 1
  - is_alone = (family_size == 1)
  - log_fare = log1p(fare)
  - title = extract from name (Mr, Mrs, Miss, Master)
  - age_group = child (<15) / adult / senior (>60)
═══════════════════════════════════════════════════════
""")

## What You Built

A complete EDA pipeline: shape inspection → missing value map → target distribution → univariate analysis of numeric and categorical features → bivariate analysis against the target → multivariate pairplot and heatmap → surprise detection → a written 5-bullet summary with feature engineering ideas. This structure transfers to any classification or regression dataset.

## What's Next

- Read `explanation.md` for the EDA mindset: why following the data's surprises matters more than running a fixed checklist.
- `howto.md` has recipes for every EDA chart type.
- `reference.md` is a chart-selection guide.
- **Extension challenge:** load seaborn's `diamonds` dataset and run the same EDA structure. What's the strongest predictor of diamond price? What would you engineer?

---

**Next module:** `10_data_science_lifecycle`, where you see how EDA fits into the end-to-end process from problem to deployed model.